In [5]:
import duckdb
import numpy as np

con = duckdb.connect("sgJobData.db")

In [7]:
# close the database connection
con.close()

In [ ]:
"""
con.sql("                                                           \
    CREATE TABLE sg_job_data AS                                     \
    SELECT * FROM read_csv_auto('data/SGJobData.csv', HEADER=TRUE); \
")
"""

# Create the table reading from the full CSV file, but add a new column listing_id 
# which is generated using the row_number() window function. This will give us 
# a unique identifier (listing_id column) for each row in the table, which we will later 
# set as the primary key.

con.sql("                                                       \
    CREATE TABLE sg_job_data AS                                 \
    SELECT                                                      \
        row_number() OVER () AS listing_id,                     \
        *                                                       \
    FROM read_csv_auto('data/SGJobData.csv', HEADER = TRUE);    \
    ")
  

In [ ]:

# Add the Primary Key constraint to the student_id column
#
## Note: execute the SQL in DBGate if it is not supported in python yet.

con.sql("                          \
    ALTER TABLE sg_job_data        \
    ADD PRIMARY KEY (listing_id);  \
    ")

In [6]:
tables = con.sql("SHOW TABLES").df()
print(tables)

for table_name in tables["name"]:
    print(f"\nSchema for table: {table_name}")
    con.sql(f"DESCRIBE {table_name}").show()
    con.sql(f"SUMMARIZE {table_name}").show()

                     name
0              categories
1  job_listing_categories
2             sg_job_data

Schema for table: categories
┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ category_id   │ INTEGER     │ NO      │ PRI     │ NULL    │ NULL    │
│ category_name │ VARCHAR     │ NO      │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌───────────────┬─────────────┬──────────────────────────────────┬─────────────────┬───────────────┬─────────┬────────────────────┬─────────┬─────────┬─────────┬───────┬─────────────────┐
│  column_name  │ column_type │               min                │       max       │ approx_unique │   avg   │        std         │   q25   │   q50   │   q75   │ count │ nul

## Normalise by extracting the data "categories" into 2 new tables:
### - job_categories
### - job-listing_categories

### test the json extract function
CREATE OR REPLACE TABLE job_listing_categories AS
SELECT DISTINCT
    j.lsiting_id,
    CAST(json_extract_string(cat.value, '$.id') AS INTEGER) AS category_id
FROM sg_job_data j,
     json_each(j.categories::JSON) AS cat;

## Normalisation: categories table
CREATE OR REPLACE TABLE categories (
    category_id INTEGER PRIMARY KEY,
    category_name VARCHAR NOT NULL
);

## Normalisation: listings to categories
CREATE OR REPLACE TABLE job_listing_categories (
    listing_id BIGINT NOT NULL,
    category_id INTEGER NOT NULL,
    PRIMARY KEY (listing_id, category_id),
    FOREIGN KEY (listing_id)
        REFERENCES sg_job_data(listing_id),
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
);

## Drop 'categories' column in original table
-- ALTER TABLE sg_job_data DROP COLUMN categories;
